# ICD-O Diagnosis Code Mapping File

## Overview
This notebook creates a mapping table that pairs ICD-O diagnosis codes with their corresponding group. This mapping file is used to assist a script that preforms the following transformation, where the user supplies a list of diagnosis codes and receives the corresponding diagnosis group information.

---
<center>
  <h3>Input Example (2 columns) → Output Example (4 columns)</h1>
</center>

<table align="center">
<tr>
<td>

| code   | term              |
|--------|-------------------|
| 8000/0 | Neoplasm, benign  |
| 8000/0 | Tumor, benign     |
| 8000/1 | Tumor, NOS        |
| 8010/0 | Epithelial tumor  | 
</td>
<td align="center" style="font-size: 24px; padding: 20px;">

**→**

</td>
<td>

| code   | term              | prefix | group           |
|--------|-------------------|--------|-----------------|
| 8000/0 | Neoplasm, benign  | 800    | Neoplasms, NOS  |
| 8000/0 | Tumor, benign     | 800    | Neoplasms, NOS  |
| 8000/1 | Tumor, NOS        | 800    | Neoplasms, NOS  |
| 8010/0 | Epithelial tumor  | 801-804 | Epithelial neoplasms, NOS |
</td>
</tr>
</table>

---

## Final Deliverable
**File:** `icdo_mapping_file.tsv`

**Format:** Tab-separated values (TSV) with the following columns:
1. `code` - (integer) The original 4-digit/1-digit diagnosis code
2. `term` - (string) The diagnosis term
3. `prefix` - (integer) The 3-digit group code or range
4. `group` - (string) The diagnosis group name

### Step 0: Data Loading

**Source:** `ICD-O-3.2_MFin_17042019_web.xlsx`
- Contains ICD-O morphology codes and terms
- Includes hierarchy levels (Level 1, 2, Preferred, Synonym)
- Level 2 rows define groups (3-digits)
- Preferred/Synonym rows contain diagnosis codes (4-digits/1-digit)

In [16]:
import pandas as pd
from IPython.display import display, Markdown

In [17]:
file_path = '../data/code_terms.xlsx'
term_codes = pd.read_excel(file_path)

In [18]:
term_codes = term_codes.iloc[:,0:3]

In [19]:
term_codes.columns

Index(['code', 'level', 'term'], dtype='object')

In [20]:
term_codes.columns = ["code","level","term"]

In [21]:
term_codes[3:6]

,code,level,term
3,8000/0,Preferred,"Neoplasm, benign"
4,8000/0,Synonym,"Tumor, benign"
5,8000/0,Synonym,"Unclassified tumor, benign"


### Step 1: Extract Group Information
Filter the data to identify diagnosis groups and their ranges.

In [22]:
term_groups = term_codes[term_codes["level"]==2]
term_groups.columns = ["range","level","group"]
term_groups.head(3)

,range,level,group
2,800,2,"Neoplasms, NOS"
34,801-804,2,"Epithelial neoplasms, NOS"
88,805-808,2,Squamous cell neoplasms


In [23]:
# avoid setting with copy warning 
term_groups = term_groups.copy() 

# split the 'range' column
delta = term_groups['range'].str.split("-", expand = True)
term_groups['range_start'] = delta[0]
term_groups['range_end'] = delta[1]
term_groups['range_end'] = term_groups['range_end'].fillna(term_groups['range_start'])
term_groups.head(5)

,range,level,group,range_start,range_end
2,800,2,"Neoplasms, NOS",800,800
34,801-804,2,"Epithelial neoplasms, NOS",801,804
88,805-808,2,Squamous cell neoplasms,805,808
185,809-811,2,Basal cell neoplasms,809,811
238,812-813,2,Transitional cell papillomas and carcinomas,812,813


### Step 3: Create Lookup Glossary
Build a reference table where each 3-digit prefix maps to its group information.

In [24]:
# expand the ranges into individual rows
expanded_rows = []
for _ , row in term_groups.iterrows():
    start = int(row['range_start'])
    end = int(row['range_end'])
    for i in range(start, end + 1):
        new_row = row.copy()
        new_row['prefix'] = i
        expanded_rows.append(new_row)
glossary = pd.DataFrame(expanded_rows)

glossary.head()

,range,level,group,range_start,range_end,prefix
2,800,2,"Neoplasms, NOS",800,800,800
34,801-804,2,"Epithelial neoplasms, NOS",801,804,801
34,801-804,2,"Epithelial neoplasms, NOS",801,804,802
34,801-804,2,"Epithelial neoplasms, NOS",801,804,803
34,801-804,2,"Epithelial neoplasms, NOS",801,804,804


### Step 4: Map Diagnosis Codes to Groups
1. Extract first 3 digits from each diagnosis code
2. Look up the group information using the glossary
3. Merge diagnosis to group information in a new dataframe

In [25]:
# select individual diagnoses (Preferred/Synonym rows)
diagnosis_df = term_codes[term_codes['level'].isin(['Preferred', 'Synonym'])].copy()
# extract first 3 digits from diagnosis code
diagnosis_df['prefix'] = diagnosis_df['code'].str[:3].astype(int)
diagnosis_df

,code,level,term,prefix
3,8000/0,Preferred,"Neoplasm, benign",800
4,8000/0,Synonym,"Tumor, benign",800
5,8000/0,Synonym,"Unclassified tumor, benign",800
6,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800
7,8000/1,Synonym,"Neoplasm, NOS",800
...,...,...,...,...
2944,9987/3,Synonym,"Therapy-related myelodysplastic syndrome, epip...",998
2945,9989/3,Preferred,"Myelodysplastic syndrome, NOS",998
2946,9989/3,Synonym,Preleukemia,998
2947,9989/3,Synonym,Preleukemic syndrome,998


In [26]:
# merge with with group info from glossary 
mapping_file = diagnosis_df.merge(
    glossary[['prefix','range','group']],
    on = 'prefix',
    how = 'left'
    )

### Step 5: Save Mapping File(s)
0. (icdo_)mapping_file is a general reference file containing level information in addition to group/diagnosis info
1. code_mapping_file is used for finding group information if the user supplies diagnosis codes
2. term_mapping_file is used for finding group information if the user supplies diagnosis terms

In [27]:
mapping_file.to_csv('../data/icdo_mapping_file.tsv', sep='\t', index=False)
mapping_file.to_csv('../data/icdo_mapping_file.csv', sep=',', index=False)
mapping_file

,code,level,term,prefix,range,group
0,8000/0,Preferred,"Neoplasm, benign",800,800,"Neoplasms, NOS"
1,8000/0,Synonym,"Tumor, benign",800,800,"Neoplasms, NOS"
2,8000/0,Synonym,"Unclassified tumor, benign",800,800,"Neoplasms, NOS"
3,8000/1,Preferred,"Neoplasm, uncertain whether benign or malignant",800,800,"Neoplasms, NOS"
4,8000/1,Synonym,"Neoplasm, NOS",800,800,"Neoplasms, NOS"
...,...,...,...,...,...,...
2326,9987/3,Synonym,"Therapy-related myelodysplastic syndrome, epip...",998,998-999,Myelodysplastic syndromes
2327,9989/3,Preferred,"Myelodysplastic syndrome, NOS",998,998-999,Myelodysplastic syndromes
2328,9989/3,Synonym,Preleukemia,998,998-999,Myelodysplastic syndromes
2329,9989/3,Synonym,Preleukemic syndrome,998,998-999,Myelodysplastic syndromes


In [28]:
# filter out synonyms
code_mapping_file = mapping_file[mapping_file['level'] == 'Preferred']
code_mapping_file = code_mapping_file.drop(columns=['level','prefix'])
# save mapping file for codes
code_mapping_file.to_csv('../data/code_mapping_file.tsv', sep='\t', index=False)
code_mapping_file.to_csv('../data/code_mapping_file.csv', sep=',', index=False)
code_mapping_file

,code,term,range,group
0,8000/0,"Neoplasm, benign",800,"Neoplasms, NOS"
3,8000/1,"Neoplasm, uncertain whether benign or malignant",800,"Neoplasms, NOS"
8,8000/3,"Neoplasm, malignant",800,"Neoplasms, NOS"
14,8000/6,"Neoplasm, metastatic",800,"Neoplasms, NOS"
19,8000/9,"Neoplasm, malignant, uncertain whether primary...",800,"Neoplasms, NOS"
...,...,...,...,...
2320,9985/3,Myelodysplastic syndrome with multilineage dys...,998-999,Myelodysplastic syndromes
2322,9986/3,Myelodysplastic syndrome with isolated del (5q),998-999,Myelodysplastic syndromes
2324,9987/3,"Therapy-related myelodysplastic syndrome, NOS",998-999,Myelodysplastic syndromes
2327,9989/3,"Myelodysplastic syndrome, NOS",998-999,Myelodysplastic syndromes


In [29]:
# save mapping file for terms
term_mapping_file = mapping_output.drop(columns=['level','prefix'])
term_mapping_file.to_csv('../data/term_mapping_file.tsv', sep='\t', index=False)
term_mapping_file.to_csv('../data/term_mapping_file.csv', sep=',', index=False)
term_mapping_file

,code,term,range,group
0,8000/0,"Neoplasm, benign",800,"Neoplasms, NOS"
1,8000/0,"Tumor, benign",800,"Neoplasms, NOS"
2,8000/0,"Unclassified tumor, benign",800,"Neoplasms, NOS"
3,8000/1,"Neoplasm, uncertain whether benign or malignant",800,"Neoplasms, NOS"
4,8000/1,"Neoplasm, NOS",800,"Neoplasms, NOS"
...,...,...,...,...
2326,9987/3,"Therapy-related myelodysplastic syndrome, epip...",998-999,Myelodysplastic syndromes
2327,9989/3,"Myelodysplastic syndrome, NOS",998-999,Myelodysplastic syndromes
2328,9989/3,Preleukemia,998-999,Myelodysplastic syndromes
2329,9989/3,Preleukemic syndrome,998-999,Myelodysplastic syndromes
